In [1]:
import pandas as pd
import torch

print("pytorch:",torch.__version__)
print("cuda: ",torch.version.cuda)
print("cuda availabale: ",torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Not detected')

pytorch: 2.14.0+cu130
cuda:  13.0
cuda availabale:  True
NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
import os
import torch
import numpy as np
BASE_DIR = r"E:\Rasengan\Cloud-Removal"

METADATA_DIR = os.path.join(BASE_DIR, "metadata")
PATCH_DIR = os.path.join(BASE_DIR, "data", "patches")

TRAIN_CSV = os.path.join(METADATA_DIR, "train.csv")
VAL_CSV = os.path.join(METADATA_DIR, "val.csv")
TEST_CSV = os.path.join(METADATA_DIR, "test.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [3]:
from torch.utils.data import DataLoader,Dataset
from PIL import Image

class CloudRemovalDataset(Dataset):
    def __init__(self,csv_file,patch_dir):
        self.df=pd.read_csv(csv_file)
        self.patch_dir=patch_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row=self.df.iloc[idx]

        cloudy_path=os.path.join(
            self.patch_dir,
            row["source"],
            "cloudy",
            row["cloudy_filename"]
        )
        clear_path=os.path.join(
            self.patch_dir,
            row["source"],
            "non-cloudy",
            row["cloudy_filename"]
        )

        cloudy=Image.open(cloudy_path).convert("RGB")
        clear=Image.open(clear_path).convert("RGB")

        cloudy=torch.from_numpy(
            np.array(cloudy)
        ).permute(2,0,1).float()/255.0

        clear=torch.from_numpy(
            np.array(clear)
        ).permute(2,0,1).float()/255.0

        return cloudy,clear

In [4]:
train_dataset=CloudRemovalDataset(TRAIN_CSV,PATCH_DIR)
val_dataset=CloudRemovalDataset(VAL_CSV,PATCH_DIR)
test_dataset=CloudRemovalDataset(TEST_CSV,PATCH_DIR)

BATCH_SIZE=16

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 11563
Validation batches: 1563
Test batches: 1563


In [5]:
cloudy_batch, clear_batch = next(iter(train_loader))

print("Cloudy batch shape:", cloudy_batch.shape)
print("Clear batch shape :", clear_batch.shape)

print("Cloudy dtype:", cloudy_batch.dtype)
print("Clear dtype :", clear_batch.dtype)

print("Cloudy range:", cloudy_batch.min().item(), "to", cloudy_batch.max().item())
print("Clear range :", clear_batch.min().item(), "to", clear_batch.max().item())

cloudy_batch = cloudy_batch.to(device)
clear_batch = clear_batch.to(device)

print("\nCloudy GPU:", cloudy_batch.device)
print("Clear GPU :", clear_batch.device)

Cloudy batch shape: torch.Size([16, 3, 256, 256])
Clear batch shape : torch.Size([16, 3, 256, 256])
Cloudy dtype: torch.float32
Clear dtype : torch.float32
Cloudy range: 0.07450980693101883 to 0.7333333492279053
Clear range : 0.003921568859368563 to 0.5764706134796143

Cloudy GPU: cuda:0
Clear GPU : cuda:0


In [9]:
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()

        self.block=nn.Sequential(
            nn.Conv2d(in_channels,out_channels,3,padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels,out_channels,3,padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self,x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1=DoubleConv(3,64)
        self.enc2=DoubleConv(64,128)
        self.enc3=DoubleConv(128,256)
        self.enc4=DoubleConv(256,512)

        self.pool=nn.MaxPool2d(2)

        self.bottleneck=DoubleConv(512,512)

        self.up4=nn.ConvTranspose2d(512,512,2,stride=2)
        self.dec4=DoubleConv(1024,512)

        self.up3=nn.ConvTranspose2d(512,256,2,stride=2)
        self.dec3=DoubleConv(512,256)

        self.up2=nn.ConvTranspose2d(256,128,2,stride=2)
        self.dec2=DoubleConv(256,128)

        self.up1=nn.ConvTranspose2d(128,64,2,stride=2)
        self.dec1=DoubleConv(128,64)

        self.output=nn.Conv2d(64,3,1)

    def forward(self,x):
        e1=self.enc1(x)
        e2=self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b=self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return torch.sigmoid(self.output(d1))

model = UNet().to(device)

print(model)

UNet(
  (enc1): DoubleConv(
    (block): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv(
    (block): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv(
    (block): Seque

In [10]:
model.eval()

with torch.no_grad():
    output = model(cloudy_batch)

print("Input shape :", cloudy_batch.shape)
print("Output shape:", output.shape)
print("Output range:", output.min().item(), "to", output.max().item())

if output.shape == clear_batch.shape:
    print("\n✓ U-Net forward pass successful.")
else:
    print("\n✗ Output shape does not match target.")

Input shape : torch.Size([16, 3, 256, 256])
Output shape: torch.Size([16, 3, 256, 256])
Output range: 0.4977402687072754 to 0.5135290026664734

✓ U-Net forward pass successful.


In [11]:
criterion = nn.L1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=2e-4
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", 2e-4)

Loss function: L1Loss()
Optimizer: Adam
Learning rate: 0.0002


In [12]:
model.train()

cloudy_batch, clear_batch = next(iter(train_loader))

cloudy_batch = cloudy_batch.to(device)
clear_batch = clear_batch.to(device)

optimizer.zero_grad()

output = model(cloudy_batch)

loss = criterion(output, clear_batch)

loss.backward()

optimizer.step()

print("Training step completed successfully.")
print("Loss:", loss.item())

Training step completed successfully.
Loss: 0.3771890103816986


In [ ]:
import time
import os

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    train_loss = 0.0
    start_time = time.time()

    for cloudy, clear in train_loader:
        cloudy = cloudy.to(device, non_blocking=True)
        clear = clear.to(device, non_blocking=True)

        optimizer.zero_grad()

        output = model(cloudy)
        loss = criterion(output, clear)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for cloudy, clear in val_loader:
            cloudy = cloudy.to(device, non_blocking=True)
            clear = clear.to(device, non_blocking=True)

            output = model(cloudy)
            loss = criterion(output, clear)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    checkpoint_path = os.path.join(
        CHECKPOINT_DIR,
        f"unet_epoch_{epoch + 1}.pth"
    )

    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss
        },
        checkpoint_path
    )

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.6f} "
        f"Val Loss: {val_loss:.6f} "
        f"Time: {elapsed / 60:.2f} min"
    )

    print(f"Checkpoint saved: {checkpoint_path}")

Epoch [1/5] Train Loss: 0.039392 Val Loss: 0.031275 Time: 442.06 min
Checkpoint saved: E:\Rasengan\Cloud-Removal\checkpoints\unet_epoch_1.pth
